In [1]:
# Install the official Kaggle API client
!pip install kaggle

In [2]:
from google.colab import files

print("Please select your 'kaggle.json' file to upload:")
files.upload()
# A file selector box will appear. Click 'Choose Files' and select the kaggle.json file.

Please select your 'kaggle.json' file to upload:


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"gangulasomashekar","key":"324fab117239ff9bc1b5060e1a2215c8"}'}

In [3]:
import os

# 1. Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# 2. Move the uploaded kaggle.json file into the .kaggle directory
!mv kaggle.json ~/.kaggle/

# 3. Set permissions: The file must be read-only for security (owner only)
!chmod 600 ~/.kaggle/kaggle.json

print("\nKaggle API Key setup complete! You are now authenticated.")

# Verify the file is in place and permissions are correct (optional)
!ls -l ~/.kaggle/


Kaggle API Key setup complete! You are now authenticated.
total 4
-rw------- 1 root root 73 Nov 11 17:48 kaggle.json


In [4]:
import os

# 1. Define the desired folder name
DOWNLOAD_PATH = './clinical_data'

# 2. Create the folder if it doesn't exist
# The -p flag in !mkdir ensures no error is thrown if the directory already exists
!mkdir -p {DOWNLOAD_PATH}
print(f"Directory '{DOWNLOAD_PATH}' created.")

# 3. Download the dataset into the specified folder using the -p argument
# The -d flag specifies the dataset, and the -p flag specifies the path.
!kaggle datasets download -d azmayensabil/doctor-patient-conversation-large -p {DOWNLOAD_PATH}
print("Download complete.")

# 4. Unzip the downloaded file inside the target folder
# The zip file will be located at: ./clinical_data/doctor-patient-conversation-large.zip
ZIP_FILE_PATH = os.path.join(DOWNLOAD_PATH, 'doctor-patient-conversation-large.zip')

# -q for quiet (optional), -d for destination directory
!unzip -q {ZIP_FILE_PATH} -d {DOWNLOAD_PATH}
print(f"Unzip complete. Files are extracted to: {DOWNLOAD_PATH}")

# 5. (Optional) List the contents of the folder to confirm the download and unzip
print("\n--- Folder Contents ---")
!ls {DOWNLOAD_PATH}

Directory './clinical_data' created.
Dataset URL: https://www.kaggle.com/datasets/azmayensabil/doctor-patient-conversation-large
License(s): unknown
  0% 0.00/786k [00:00<?, ?B/s]
100% 786k/786k [00:00<00:00, 1.06GB/s]
Download complete.
Unzip complete. Files are extracted to: ./clinical_data

--- Folder Contents ---
CAR0001.txt			       RES0010.txt  RES0081.txt  RES0151.txt
CAR0002.txt			       RES0011.txt  RES0082.txt  RES0152.txt
CAR0003.txt			       RES0012.txt  RES0083.txt  RES0153.txt
CAR0004.txt			       RES0013.txt  RES0084.txt  RES0154.txt
CAR0005.txt			       RES0014.txt  RES0085.txt  RES0155.txt
DER0001.txt			       RES0015.txt  RES0086.txt  RES0156.txt
doctor-patient-conversation-large.zip  RES0016.txt  RES0087.txt  RES0158.txt
GAS0001.txt			       RES0017.txt  RES0088.txt  RES0159.txt
GAS0002.txt			       RES0018.txt  RES0089.txt  RES0160.txt
GAS0003.txt			       RES0019.txt  RES0090.txt  RES0161.txt
GAS0004.txt			       RES0020.txt  RES0091.txt  RES0162.txt
GAS0005.txt			

In [5]:
pip install transformers torch datasets seqeval scikit-learn pandas tqdm evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=390a4cb217723a1e426a290087ae02087bc640c8cfd1ea8bf84e245a6244ffdd
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [6]:
import os
import sys
import json
import re
import argparse
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict, Counter
from datetime import datetime
from dataclasses import dataclass
from tqdm import tqdm
import pandas as pd

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Check imports
try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForTokenClassification,
        pipeline
    )
except ImportError as e:
    logger.error(f"Missing dependencies: {e}")
    print("❌ Missing dependencies. Please install:")
    print("\npip install transformers torch pandas tqdm\n")
    sys.exit(1)

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class ModelConfig:
    """Model configuration parameters"""
    # Using a different, more reliable medical NER model
    model_name: str = "d4data/biomedical-ner-all"
    # Alternative: "bvanaken/clinical-ner-biobert"
    max_length: int = 512
    batch_size: int = 4  # Reduced for stability

class Config:
    def __init__(self, data_dir: str):
        self.project_root = Path.cwd()
        self.raw_data_dir = Path(data_dir)
        self.output_dir = self.project_root / "outputs"
        self.model_dir = self.project_root / "models"

        # Create directories
        self.output_dir.mkdir(exist_ok=True)
        self.model_dir.mkdir(exist_ok=True)
        (self.output_dir / "summaries").mkdir(exist_ok=True)
        (self.output_dir / "entities").mkdir(exist_ok=True)
        (self.output_dir / "statistics").mkdir(exist_ok=True)
        (self.output_dir / "reports").mkdir(exist_ok=True)

        self.model = ModelConfig()

# ============================================================================
# IMPROVED CONVERSATION PARSER
# ============================================================================

class ConversationParser:
    def __init__(self, config: Config):
        self.config = config

    def parse_all_files(self, data_dir: Path) -> List[Dict]:
        """Parse all conversation files with improved filtering"""
        files = sorted(list(data_dir.glob('*.txt')))

        conversations = []
        logger.info(f"📁 Parsing {len(files)} conversation files...")

        for filepath in tqdm(files, desc="Parsing files"):
            conversation = self.parse_file(filepath)
            if conversation['num_turns'] > 0:
                conversations.append(conversation)

        logger.info(f"✅ Successfully parsed {len(conversations)} conversations")
        return conversations

    def parse_file(self, filepath: Path) -> Dict:
        """Parse a single conversation file"""
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read().strip()

            if not content:
                return self._create_empty_conversation(filepath)

            lines = [line.strip() for line in content.split('\n') if line.strip()]
            turns = self._parse_lines(lines)

            return {
                'filename': filepath.name,
                'filepath': str(filepath),
                'turns': turns,
                'num_turns': len(turns)
            }

        except Exception as e:
            logger.error(f"Error parsing {filepath}: {e}")
            return self._create_empty_conversation(filepath)

    def _parse_lines(self, lines: List[str]) -> List[Dict]:
        """Parse individual lines into conversation turns"""
        turns = []
        current_speaker = None
        current_text = []

        for line in lines:
            speaker, text = self._parse_line(line)

            if speaker:
                if current_speaker and current_text:
                    turns.append({
                        'speaker': current_speaker,
                        'text': ' '.join(current_text)
                    })
                current_speaker = speaker
                current_text = [text] if text else []
            elif current_speaker and text:
                current_text.append(text)

        if current_speaker and current_text:
            turns.append({
                'speaker': current_speaker,
                'text': ' '.join(current_text)
            })

        return turns

    def _parse_line(self, line: str) -> Tuple[Optional[str], Optional[str]]:
        """Parse a single line for speaker and text"""
        line = line.strip()

        if line.startswith('D:'):
            return 'doctor', line[2:].strip()
        elif line.startswith('P:'):
            return 'patient', line[2:].strip()
        elif line:
            return None, line
        else:
            return None, None

    def _create_empty_conversation(self, filepath: Path) -> Dict:
        return {
            'filename': filepath.name,
            'filepath': str(filepath),
            'turns': [],
            'num_turns': 0
        }

# ============================================================================
# GREATLY IMPROVED ENTITY POST-PROCESSOR
# ============================================================================

class AdvancedClinicalEntityPostProcessor:
    """Much more sophisticated entity cleaning and categorization"""

    def __init__(self):
        # Comprehensive medical dictionaries
        self.medical_terms = {
            'SYMPTOM': {
                'cough', 'fever', 'headache', 'nausea', 'vomiting', 'diarrhea', 'fatigue',
                'pain', 'sore throat', 'runny nose', 'congestion', 'sneezing', 'wheezing',
                'shortness of breath', 'chest pain', 'abdominal pain', 'body aches',
                'chills', 'sweating', 'rash', 'itching', 'dizziness', 'weakness',
                'hoarseness', 'loss of voice', 'muscle pain', 'joint pain', 'back pain',
                'neck pain', 'swelling', 'redness', 'bleeding', 'bruising', 'numbness',
                'tingling', 'palpitations', 'heartburn', 'indigestion', 'constipation',
                'bloating', 'loss of appetite', 'weight loss', 'weight gain',
                'insomnia', 'anxiety', 'depression', 'confusion', 'memory loss'
            },
            'MEDICATION': {
                'aspirin', 'ibuprofen', 'acetaminophen', 'tylenol', 'advil', 'aleve',
                'antibiotics', 'penicillin', 'amoxicillin', 'azithromycin', 'doxycycline',
                'inhaler', 'albuterol', 'steroids', 'prednisone', 'insulin', 'metformin',
                'lisinopril', 'atorvastatin', 'simvastatin', 'omeprazole', 'prilosec',
                'levothyroxine', 'synthroid', 'warfarin', 'coumadin', 'clopidogrel',
                'plavix', 'metoprolol', 'losartan', 'amlodipine', 'hydrochlorothiazide',
                'multivitamin', 'vitamin d', 'calcium', 'iron', 'fish oil'
            },
            'DIAGNOSIS': {
                'covid', 'coronavirus', 'influenza', 'flu', 'cold', 'pneumonia',
                'bronchitis', 'sinusitis', 'asthma', 'copd', 'diabetes', 'hypertension',
                'high blood pressure', 'heart disease', 'arthritis', 'migraine',
                'anxiety', 'depression', 'cancer', 'infection', 'viral infection',
                'bacterial infection', 'uti', 'urinary tract infection', 'gerd',
                'acid reflux', 'allergies', 'anemia', 'thyroid disorder'
            },
            'TEST': {
                'blood test', 'x-ray', 'mri', 'ct scan', 'ultrasound', 'ekg', 'ecg',
                'stress test', 'pulmonary function test', 'urinalysis', 'biopsy',
                'endoscopy', 'colonoscopy', 'mammogram', 'pap smear', 'covid test',
                'pcr test', 'rapid test', 'allergy test', 'glucose test'
            },
            'BODY_PART': {
                'head', 'neck', 'chest', 'back', 'abdomen', 'stomach', 'arm', 'leg',
                'hand', 'foot', 'heart', 'lung', 'liver', 'kidney', 'brain', 'throat',
                'nose', 'ear', 'eye', 'skin', 'bone', 'muscle', 'joint', 'blood'
            }
        }

        # Common phrases to exclude
        self.exclude_phrases = {
            'any', 'some', 'the', 'a', 'this', 'that', 'these', 'those', 'my', 'your',
            'have you', 'do you', 'did you', 'are you', 'is there', 'any other',
            'similar symptoms', 'medical conditions', 'heart or lung conditions',
            'urinary problems', 'sense of smell', 'skin changes', 'weight gain',
            'night sweats', 'eye redness', 'eye discharge', 'nasal congestion',
            'physical exam', 'family doctor', 'hospitalizations', 'surgeries'
        }

        # Medical term variations mapping
        self.term_variations = {
            'coughing': 'cough',
            'vomiting': 'vomit',
            'nauseous': 'nausea',
            'aching': 'pain',
            'hurting': 'pain',
            'soreness': 'pain',
            'hoarse': 'hoarseness',
            'runny': 'runny nose',
            'stuffy': 'nasal congestion',
            'wheeze': 'wheezing',
            'shortness of breath': 'breathing difficulty',
            'high blood pressure': 'hypertension'
        }

    def clean_and_categorize_entities(self, raw_entities: List[Dict], text: str) -> List[Dict]:
        """Greatly improved entity cleaning and categorization"""
        if not raw_entities:
            return []

        # First pass: basic cleaning
        initially_cleaned = []
        for entity in raw_entities:
            entity_text = self._basic_clean_entity(entity['word'])

            # Skip if too short or excluded
            if (len(entity_text) < 3 or
                entity_text.lower() in self.exclude_phrases or
                any(excluded in entity_text.lower() for excluded in self.exclude_phrases if len(excluded) > 3)): # Added check for longer excluded phrases
                continue

            initially_cleaned.append({
                'text': entity_text,
                'score': float(entity['score']),
                'original_type': entity['entity_group']
            })

        # Second pass: advanced categorization and filtering
        final_entities = []
        for entity in initially_cleaned:
            # Skip low confidence entities
            if entity['score'] < 0.7:
                continue

            entity_text = entity['text']
            entity_lower = entity_text.lower()

            # Apply term variations
            if entity_lower in self.term_variations:
                entity_text = self.term_variations[entity_lower]
                entity_lower = entity_text.lower()

            # Categorize using our medical dictionaries
            entity_type = self._categorize_entity(entity_text, entity_lower)

            if entity_type:
                final_entities.append({
                    'word': entity_text.title() if len(entity_text) > 3 else entity_text, # Capitalize for better readability
                    'entity_group': entity_type,
                    'score': entity['score'],
                    'confidence': entity['score']
                })

        # Remove duplicates
        return self._remove_duplicates(final_entities)

    def _basic_clean_entity(self, text: str) -> str:
        """Basic cleaning of entity text"""
        cleaned = text.replace('##', '').strip()
        cleaned = re.sub(r'[^​-\u200D\uFEFF\w\s]', '', cleaned) # Remove punctuation, preserving some unicode spaces
        return cleaned.strip()

    def _categorize_entity(self, entity_text: str, entity_lower: str) -> str:
        """Categorize entity using medical dictionaries"""
        # Check each category
        for category, terms in self.medical_terms.items():
            # Check exact matches first
            if entity_lower in terms:
                return category

            # Check for partial matches (for multi-word terms)
            for term in terms:
                if term in entity_lower or entity_lower in term:
                    # Only return if it's a meaningful match (avoid matching 'a' in 'pain')
                    if len(term) > 3 and len(entity_lower) > 3: # Both terms should be reasonably long
                        return category

        # If no match found, use some heuristics
        if any(symptom in entity_lower for symptom in ['pain', 'ache', 'fever', 'cough', 'nausea', 'breath', 'hoarse']):
            return 'SYMPTOM'
        elif any(med in entity_lower for med in ['antibiotic', 'medication', 'pill', 'tablet', 'insulin']):
            return 'MEDICATION'
        elif any(test in entity_lower for test in ['test', 'scan', 'xray', 'mri', 'ekg']):
            return 'TEST'
        elif any(part in entity_lower for part in ['chest', 'head', 'throat', 'lung', 'heart', 'abdomen', 'leg', 'arm']):
            return 'BODY_PART'
        elif any(diag in entity_lower for diag in ['infection', 'diabetes', 'hypertension', 'cancer']):
            return 'DIAGNOSIS'

        return None  # Skip if we can't categorize reliably

    def _remove_duplicates(self, entities: List[Dict]) -> List[Dict]:
        """Remove duplicate entities"""
        seen = set()
        unique_entities = []

        for entity in entities:
            key = (entity['word'].lower(), entity['entity_group'])
            if key not in seen:
                seen.add(key)
                unique_entities.append(entity)

        return unique_entities

# ============================================================================
# IMPROVED CLINICAL NER PIPELINE
# ============================================================================

class ImprovedClinicalNERPipeline:
    """Improved clinical NER pipeline with better entity filtering"""

    def __init__(self, config: Config):
        self.config = config
        self.postprocessor = AdvancedClinicalEntityPostProcessor()
        self.ner_pipeline = None
        self._load_model()

    def _load_model(self):
        """Load the pre-trained clinical NER model"""
        try:
            logger.info(f"🚀 Loading clinical NER model: {self.config.model.model_name}")

            self.ner_pipeline = pipeline(
                "ner",
                model=self.config.model.model_name,
                tokenizer=self.config.model.model_name,
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1,
                batch_size=self.config.model.batch_size
            )

            logger.info("✅ Clinical NER model loaded successfully!")

        except Exception as e:
            logger.error(f"❌ Error loading model: {e}")
            logger.info("🔄 Trying BioClinicalBERT as fallback...")

            self.config.model.model_name = "emilyalsentzer/Bio_ClinicalBERT"
            self.ner_pipeline = pipeline(
                "ner",
                model=self.config.model.model_name,
                tokenizer=self.config.model.model_name,
                aggregation_strategy="simple",
                device=0 if torch.cuda.is_available() else -1
            )

    def extract_clinical_entities(self, conversation: Dict) -> Dict:
        """Extract clinical entities with much better filtering"""
        all_entities = []

        for turn in conversation['turns']:
            if not turn['text'].strip() or len(turn['text'].strip()) < 10: # Minimum text length for processing
                continue

            try:
                # Extract entities from this turn
                raw_entities = self.ner_pipeline(turn['text'])

                # Clean and categorize entities
                cleaned_entities = self.postprocessor.clean_and_categorize_entities(
                    raw_entities, turn['text']
                )

                # Add conversation context
                for entity in cleaned_entities:
                    entity['speaker'] = turn['speaker']
                    entity['turn_text'] = turn['text'][:80] + '...' if len(turn['text']) > 80 else turn['text']

                all_entities.extend(cleaned_entities)

            except Exception as e:
                logger.warning(f"⚠️ Error processing turn in {conversation['filename']}: {str(e)[:100]}...")
                continue # Skip errors silently to process other turns

        return {
            'filename': conversation['filename'],
            'entities': all_entities,
            'num_entities': len(all_entities),
            'speaker_breakdown': self._get_speaker_breakdown(all_entities)
        }

    def extract_entities_batch(self, conversations: List[Dict]) -> List[Dict]:
        """Extract entities from multiple conversations"""
        all_results = []

        logger.info(f"🔍 Extracting clinical entities from {len(conversations)} conversations...")

        for conversation in tqdm(conversations, desc="Processing conversations"):
            result = self.extract_clinical_entities(conversation)
            all_results.append(result)

        return all_results

    def _get_speaker_breakdown(self, entities: List[Dict]) -> Dict[str, int]:
        """Get entity count by speaker"""
        breakdown = defaultdict(int)
        for entity in entities:
            breakdown[entity.get('speaker', 'unknown')] += 1
        return dict(breakdown)

# ============================================================================
# CLEAN RESULTS GENERATOR
# ============================================================================

class CleanResultsGenerator:
    """Generate clean, meaningful clinical summaries"""

    @staticmethod
    def generate_detailed_results(extraction_result: Dict) -> Dict:
        """Generate detailed, organized results"""
        entities = extraction_result['entities']

        # Group entities by type
        symptoms = []
        medications = []
        diagnoses = []
        tests = []
        body_parts = []

        for entity in entities:
            entity_type = entity['entity_group']
            entity_text = entity['word']

            # Additional filtering for quality
            if len(entity_text) < 4:  # Skip very short entities like 'A' or 'The'
                continue

            if entity_type == 'SYMPTOM':
                symptoms.append(entity_text)
            elif entity_type == 'MEDICATION':
                medications.append(entity_text)
            elif entity_type == 'DIAGNOSIS':
                diagnoses.append(entity_text)
            elif entity_type == 'TEST':
                tests.append(entity_text)
            elif entity_type == 'BODY_PART':
                body_parts.append(entity_text)

        # Remove duplicates and sort for consistency
        symptoms = sorted(list(set(symptoms)))
        medications = sorted(list(set(medications)))
        diagnoses = sorted(list(set(diagnoses)))
        tests = sorted(list(set(tests)))
        body_parts = sorted(list(set(body_parts)))

        return {
            'symptoms': symptoms,
            'medications': medications,
            'diagnoses': diagnoses,
            'tests': tests,
            'body_parts': body_parts,
            'summary': {
                'total_entities': len(symptoms) + len(medications) + len(diagnoses) + len(tests) + len(body_parts),
                'symptoms_count': len(symptoms),
                'medications_count': len(medications),
                'diagnoses_count': len(diagnoses),
                'tests_count': len(tests),
                'body_parts_count': len(body_parts)
            }
        }

    @staticmethod
    def generate_clinical_summary(detailed_results: Dict, filename: str) -> str:
        """Generate a clean, professional clinical summary"""
        summary = []
        summary.append("=" * 70)
        summary.append("CLINICAL ENTITY EXTRACTION SUMMARY")
        summary.append("=" * 70)
        summary.append(f"File: {filename}")
        summary.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        summary.append("")

        # Patient Symptoms
        summary.append("🧬 PATIENT SYMPTOMS:")
        summary.append("-" * 40)
        symptoms = detailed_results['symptoms']
        if symptoms:
            for symptom in symptoms[:15]:  # Limit to top 15 for brevity in summary
                summary.append(f"  • {symptom}")
            if len(symptoms) > 15:
                summary.append(f"  • ... and {len(symptoms) - 15} more")
        else:
            summary.append("  No significant symptoms identified")

        # Medications
        summary.append("\n💊 MEDICATIONS & TREATMENTS:")
        summary.append("-" * 40)
        medications = detailed_results['medications']
        if medications:
            for med in medications:
                summary.append(f"  • {med}")
        else:
            summary.append("  No medications identified")

        # Clinical Diagnoses
        summary.append("\n🏥 CLINICAL IMPRESSIONS:")
        summary.append("-" * 40)
        diagnoses = detailed_results['diagnoses']
        if diagnoses:
            for diag in diagnoses:
                summary.append(f"  • {diag}")
        else:
            summary.append("  No specific diagnoses mentioned")

        # Diagnostic Tests
        summary.append("\n🔬 DIAGNOSTIC TESTS:")
        summary.append("-" * 40)
        tests = detailed_results['tests']
        if tests:
            for test in tests:
                summary.append(f"  • {test}")
        else:
            summary.append("  No tests mentioned")

        # Body Parts
        summary.append("\n📍 BODY PARTS MENTIONED:")
        summary.append("-" * 40)
        body_parts = detailed_results['body_parts']
        if body_parts:
            for part in body_parts:
                summary.append(f"  • {part}")
        else:
            summary.append("  No specific body parts mentioned")

        # Statistics
        summary.append("\n📊 EXTRACTION STATISTICS:")
        summary.append("-" * 40)
        stats = detailed_results['summary']
        summary.append(f"  Total meaningful entities: {stats['total_entities']}")
        summary.append(f"  Symptoms: {stats['symptoms_count']}")
        summary.append(f"  Medications: {stats['medications_count']}")
        summary.append(f"  Diagnoses: {stats['diagnoses_count']}")
        summary.append(f"  Tests: {stats['tests_count']}")
        summary.append(f"  Body parts: {stats['body_parts_count']}")

        summary.append("=" * 70)
        return "\n".join(summary)

# ============================================================================
# MAIN PIPELINE
# ============================================================================

class CleanMedicalSLUPipeline:
    """Main pipeline with clean entity extraction"""

    def __init__(self, data_dir: str):
        self.config = Config(data_dir)

    def run(self):
        """Execute the pipeline"""
        logger.info("🚀 Starting Clean Medical Entity Extraction Pipeline")
        start_time = datetime.now()

        try:
            # Step 1: Parse conversations
            conversations = self.step_parse_conversations()

            # Step 2: Extract entities with improved processing
            extraction_results = self.step_extract_entities(conversations)

            # Step 3: Generate clean summaries
            self.step_generate_outputs(extraction_results)

            # Final summary
            self.generate_final_summary(start_time, extraction_results)

        except Exception as e:
            logger.error(f"Pipeline execution failed: {e}")
            raise

    def step_parse_conversations(self) -> List[Dict]:
        """Step 1: Parse conversations"""
        parser = ConversationParser(self.config)
        return parser.parse_all_files(self.config.raw_data_dir)

    def step_extract_entities(self, conversations: List[Dict]) -> List[Dict]:
        """Step 2: Extract entities"""
        ner_pipeline = ImprovedClinicalNERPipeline(self.config)
        return ner_pipeline.extract_entities_batch(conversations)

    def step_generate_outputs(self, extraction_results: List[Dict]):
        """Step 3: Generate outputs"""
        logger.info("📊 Generating clean clinical summaries...")

        for result in tqdm(extraction_results, desc="Creating summaries"):
            detailed_results = CleanResultsGenerator.generate_detailed_results(result)
            clinical_summary = CleanResultsGenerator.generate_clinical_summary(
                detailed_results, result['filename']
            )

            filename = Path(result['filename']).stem
            output_file = self.config.output_dir / "summaries" / f"{filename}_clinical_summary.txt"

            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(clinical_summary)

    def generate_final_summary(self, start_time: datetime, extraction_results: List[Dict]):
        """Generate final summary"""
        end_time = datetime.now()
        duration = end_time - start_time

        total_entities = sum(r['num_entities'] for r in extraction_results)
        total_conversations = len(extraction_results)

        summary = f"""
╔{'═' * 68}╗
║              CLEAN EXTRACTION COMPLETE                 ║
╚{'═' * 68}╝

📊 RESULTS:
  • Conversations processed: {total_conversations}
  • Clean entities extracted: {total_entities}
  • Average per conversation: {total_entities/total_conversations:.1f}
  • Execution time: {duration}

📁 OUTPUTS:
  • Clean Summaries: {self.config.output_dir / 'summaries'}

✅ PIPELINE COMPLETED SUCCESSFULLY!
"""
        logger.info(summary)

# ============================================================================
# EXECUTION
# ============================================================================

def main():
    """Main execution"""
    parser = argparse.ArgumentParser(description='Clean Medical Entity Extraction')
    parser.add_argument('/content/clinical_data', type=str,
                       help='Path to directory containing clinical conversation .txt files')

    args = parser.parse_args([DOWNLOAD_PATH])

    if not Path(args.data_dir).exists():
        print(f"❌ Error: Data directory '{args.data_dir}' does not exist")
        sys.exit(1)

    try:
        pipeline = CleanMedicalSLUPipeline(args.data_dir)
        pipeline.run()
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

Parsing files: 100%|██████████| 272/272 [00:00<00:00, 6428.78it/s]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0
Creating summaries: 100%|██████████| 270/270 [00:00<00:00, 7127.56it/s]
